<a href="https://colab.research.google.com/github/spatra12/CS690U/blob/main/task1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**: We are fine-tuning a DNABERT-2 model to predict which alleles of the IL2RA gene—whose lack of expression is a key
cause of Lupus—are high risk for the disease [3], [2]. We developed a fine-tuned DNABERT-2 framework [6] and [1]. The
model evaluates risk through a four-tier analysis: (1) Motif analysis via PWMs to assess binding site integrity [4], [5]; (2)
semantic context analysis to evaluate the biophysical properties surrounding the motif (DNA flexibility) [8]; (3) prediction of
DNA ’openness’ based on ATAC-seq data (chromatin accessibility) [7]; and (4) structural connectivity to model long-range
Enhancer-Promoter loops [9] and [10]. This multi-tiered approach allows the model to detect not only direct disruptions of binding sites but also the indirect structural failures that drive immune dysregulation in Lupus patients.

Task 1. Classification Head with DNABERT-2.
Model: Pre-trained DNABERT-2 with an additional linear layer of classification.

*   The model will be downloaded from Hugging
Face.
*   Method: Supervised classification of Transcription Factor (TF) binding sites. The model will be trained using ChIP-seq
peaks (from ENCODE) and PWM scoring (from JASPAR) from healthy samples. Subsequently, the original sequences
will be modified to include the Lupus pathogenic variants identified by Farh et al. By calculating and comparing the scores
of both the healthy (wild-type) and pathogenic sequences, the model will quantify the associated risk of each variant.
*   Implementations: We will work at the embedding level with genomic data and will use sklearn.svm.SVC or
sklearn.linear model.LogisticRegression for the final classification. The embedding will be building using
transformers. The input X is an (N, 768) matrix, and the output yˆ is a class vector of size N, with N the instances
number. We will use F1-score (if we have unbalance classes) and confusion matrix. As a metric of biological impact
(Effects of mutation), we can use cosine similarity at the embedding level. The true label (y) will be obtained of ChIP-seq
peaks.The enviroment will be Google Colab, it allows for work with GPU NVIDIA T4 in a free way. The hyperparameters
for tuning are learning rate, sequencing window (for example 512), weight decay, dimension of the vectors for each token,
vocabulary size, attention heads.

In [1]:
!pip install transformers torch einops triton

In [2]:
!pip install scikit-learn biopython pyfaidx pybedtools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 42.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 58.8 MB/s eta 0:00:00
  Created wheel for pybedtools: filename=pybedtools-0.12.0-cp312-cp312-linux_x86_64.whl size=14340827 sha256=07d5fef70690d32f3e6d3c739a1b37e4173da3b19f6818f8834bd6bdb69aec96
  Stored in directory: /root/.cache/pip/wheels/ac/38/f2/960d79e44a92afc0d34a4727c856ce0149ac23c3dcda174356
Successfully built pybedtools


In [3]:
!pip install biopython

In [5]:
!git config --global user.email "samarpita.patra@gmail.com"
!git config --global user.name "spatra12"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
!cat chip_seqs/files_elf1.txt | xargs -n 1 curl -O -L

cat: chip_seqs/files_elf1.txt: No such file or directory
curl: no URL specified!
curl: try 'curl --help' or 'curl --manual' for more information


In [ ]:
!cat chip_seqs/files_stat5a_stat5b.txt | xargs -n 1 curl -O -L

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: Failed to create the file 
100  1119    0  1119    0     0   4403      0 --:--:-- --:--:-- --:--:--  4422
curl: (23) Failure writing output to destination
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1913  100  1913    0     0   8597      0 --:--:-- --:--:-- --:--:--  8617
100 1125M  100 1125M    0     0   9.9M      0  0:01:53  0:01:53 --:--:-- 4055k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1915  100  1915    0     0   6256      0 --:--:-- --:--:-- --:--:--  6258
100  150k  100  150k    0     0   211k      0 -

In [ ]:
!zip -r chip_seqs.zip /content/chip_seqs/

  adding: content/chip_seqs/ (stored 0%)
  adding: content/chip_seqs/ENCFF119COY.bigWig (deflated 3%)
  adding: content/chip_seqs/ENCFF653KLN.bigWig (deflated 2%)
  adding: content/chip_seqs/ENCFF006GSY.bed.gz (deflated 0%)
  adding: content/chip_seqs/ENCFF370XOM.bed.gz (deflated 0%)
  adding: content/chip_seqs/files_elf1.txt (deflated 40%)
  adding: content/chip_seqs/ENCFF116OUV.bigBed (deflated 22%)
  adding: content/chip_seqs/files_stat5a_stat5b.txt (deflated 58%)
  adding: content/chip_seqs/ENCFF267JUM.bigBed (deflated 15%)
  adding: content/chip_seqs/ENCFF415UIB.bigWig (deflated 3%)
  adding: content/chip_seqs/ENCFF692SMY.bigBed (deflated 5%)


In [ ]:
!unzip /content/chip_seqs.zip -d /content/chip_seqs

In [ ]:
!mkdir -p genome
!wget -O genome/chr10.fa.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
!gunzip genome/chr10.fa.gz

--2026-05-03 18:43:26--  https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 43157332 (41M) [application/x-gzip]
Saving to: ‘genome/chr10.fa.gz’

genome/chr10.fa.gz  100%[===================>]  41.16M   101MB/s    in 0.4s    

2026-05-03 18:43:26 (101 MB/s) - ‘genome/chr10.fa.gz’ saved [43157332/43157332]



In [5]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [6]:
import torch
from transformers import AutoTokenizer, AutoModel
from transformers.models.bert.configuration_bert import BertConfig
from Bio import motifs
from pyfaidx import Fasta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
import numpy as np
from sklearn.model_selection import train_test_split

import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression


In [7]:
#SANITY CHECK DNABERT-2 MODEL loading

#original model loading was not working

# tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
# model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
# inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
# hidden_states = model(inputs)[0] # [1, sequence_length, 768]

# # embedding with mean pooling
# embedding_mean = torch.mean(hidden_states[0], dim=0)
# print(embedding_mean.shape) # expect to be 768

# # embedding with max pooling
# embedding_max = torch.max(hidden_states[0], dim=0)[0]
# print(embedding_max.shape) # expect to be 768

#fix from https://huggingface.co/quietflamingo/dnabert2-no-flashattention

print("=" * 60)
print("DNABERT-2 Model Testing")
print("=" * 60)

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✓ Device: {device}")

# Load model
print("\nLoading DNABERT-2...")
tokenizer = AutoTokenizer.from_pretrained(
    "quietflamingo/dnabert2-fixed",
    trust_remote_code=True
)

config = BertConfig.from_pretrained("quietflamingo/dnabert2-fixed")
model = AutoModel.from_pretrained("quietflamingo/dnabert2-fixed", config=config)
model = model.to(device)

if torch.cuda.is_available():
    model = model.half()

print(f"✓ Model loaded on: {device}")


DNABERT-2 Model Testing

✓ Device: cuda

Loading DNABERT-2...


config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

configuration_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-fixed:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

BertModel LOAD REPORT from: quietflamingo/dnabert2-fixed
Key                                                    | Status     | 
-------------------------------------------------------+------------+-
bert.encoder.layer.{0...11}.attention.self.Wqkv.weight | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.layernorm.bias         | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.gated_layers.weight    | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.layernorm.weight       | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.wo.bias                | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.wo.weight              | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.Wqkv.bias   | UNEXPECTED | 
cls.predictions.decoder.weight                         | UNEXPECTED | 
cls.predictions.transform.dense.weight                 | UNEXPECTED | 
cls.predictions.transform.dense.bias                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias               | UNEXPECTED | 
cls.predictions.tran

✓ Model loaded on: cuda


In [8]:
### VERIFY model can generate embeddings

dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
# Tokenize - this returns a dictionary
inputs = tokenizer(dna, return_tensors='pt', padding=True, truncation=True)

# Check what inputs contains
print(f"Input keys: {inputs.keys()}")
print(f"Input IDs shape: {inputs['input_ids'].shape}")

# Move to device
inputs = {k: v.to(device) for k, v in inputs.items()}

hidden_states = model(**inputs)[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape) # expect to be 768

# embedding with max pooling
embedding_max = torch.max(hidden_states[0], dim=0)[0]
print(embedding_max.shape) # expect to be 768


Input keys: KeysView({'input_ids': tensor([[   1,    5,  194,   32,  757, 1239, 2092,  294,   24,  359,   88,   93,
           32,   75,   77,   19,    2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
Input IDs shape: torch.Size([1, 17])
torch.Size([768])
torch.Size([768])


Download PFM data and calculate PWM for NFkB TF

In [9]:
filepath = "/content/drive/MyDrive/Colab Notebooks/cs690u_data/pfm_matrices/MA0105.4.pfm"
with open(filepath) as handle:
  motif = motifs.read(handle, "jaspar")

nfkb_pwm = motif.pwm

print(nfkb_pwm)

        0      1      2      3      4      5      6      7      8      9     10     11     12
A:   0.69   0.00   0.00   0.00   0.01   0.93   0.51   0.00   0.00   0.05   0.00   0.00   0.08
C:   0.16   0.00   0.00   0.00   0.00   0.00   0.00   0.08   0.98   0.95   1.00   1.00   0.08
G:   0.08   1.00   1.00   0.98   0.99   0.06   0.00   0.01   0.00   0.00   0.00   0.00   0.22
T:   0.07   0.00   0.00   0.02   0.00   0.00   0.49   0.91   0.01   0.00   0.00   0.00   0.62



Download PFM data and calculate PWM for ELF1 TF

In [10]:
filepath = "/content/drive/MyDrive/Colab Notebooks/cs690u_data/pfm_matrices/MA0473.3.pfm"
with open(filepath) as handle:
  motif = motifs.read(handle, "jaspar")

elf1_pwm = motif.pwm

print(elf1_pwm)

        0      1      2      3      4      5      6      7      8      9     10     11     12     13
A:   0.39   0.39   0.37   0.06   0.88   0.01   0.01   0.96   0.95   0.09   0.06   0.12   0.26   0.25
C:   0.18   0.16   0.27   0.79   0.08   0.01   0.01   0.02   0.01   0.03   0.04   0.09   0.25   0.27
G:   0.24   0.24   0.22   0.10   0.02   0.98   0.96   0.01   0.02   0.87   0.04   0.73   0.35   0.28
T:   0.19   0.21   0.15   0.04   0.01   0.01   0.01   0.01   0.02   0.01   0.85   0.06   0.13   0.20



In [7]:
genome_chr10 = Fasta('/content/drive/MyDrive/Colab Notebooks/cs690u_data/genome/chr10.fa')

In [8]:
!pip install -q pandas matplotlib seaborn
!wget -q http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/bigBedToBed
!chmod +x bigBedToBed

In [ ]:
print("\nConverting bigBed to BED format...")
!./bigBedToBed \
    "/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/ENCFF692SMY.bigBed" \
    "/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/ELF1_peaks.bed"


Converting bigBed to BED format...


In [25]:
print("\nConverting bigBed to BED format...")
!./bigBedToBed \
    "/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/ENCFF267JUM.bigBed" \
    "/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/STAT5A_peaks.bed"


Converting bigBed to BED format...


In [26]:
print("\nConverting bigBed to BED format...")
!./bigBedToBed \
    "/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/ENCFF116OUV.bigBed" \
    "/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/STAT5B_peaks.bed"


Converting bigBed to BED format...


ChIP_seq data gives you:
  chr (chromosome number),
  start (start of tf binding on human genome seq),
  end (end of tf binding on human genome seq),
  score (scaled quality value, use data above score >= 700 ),
  signalValue (raw ChIP-seq enrichment),
  pValue (pValue == -log10(p-value)),
  qValue (qValue == -log10(q-value)),
  width (length of TF binding to calculate sequence segments)

In [8]:
#source: https://www.ncbi.nlm.nih.gov/gene/3559#genomic-context
ilra_gene_coordinates = [6010689, 6062370] #10:6010689-6062370

In [9]:
def generate_allele_sequence_dataset_for_wt_gene(filepaths):

  wt_sequence_df = pd.DataFrame(columns=[
      'tf_name',
      'marker',
      'genome_range',
      'seq',
      'score',
      'signalValue',
      'is_binding'])

  all_pos_sequences = []
  all_neg_sequences = []

  for tf_name, filepath in filepaths.items():
    print(f"\nAnalyzing peaks of {tf_name}...")

    # Read peaks
    peaks = pd.read_csv(filepath, sep='\t', header=None)
    peaks.columns = ['chr', 'start', 'end', 'name', 'score', 'strand',
                    'signalValue', 'pValue', 'qValue', 'peak']
    peaks['width'] = peaks['end'] - peaks['start']

    peaks_chr10 = peaks[peaks['chr'] == 'chr10']

    # Print summary
    print(f"\n{'='*60}")
    print(f"{tf_name} in Chr10 ChIP-seq Summary")
    print(f"{'='*60}")
    print(f"Total peaks: {len(peaks_chr10):,}")
    print(f"Mean peak width: {peaks_chr10['width'].mean():.1f} bp")

    #Positive sequences that correlate to TF binding
    records = []
    window_size = 512 #to match DNABERT-2 max tokens

    for measured_peak in peaks_chr10.itertuples():

      window_start = measured_peak.start - 50
      window_end = window_start + window_size
      seq_obj = genome_chr10['chr10'][window_start:window_end]

      if measured_peak.score >= 700:
        records.append({
            'tf_name': tf_name,
            'marker': seq_obj.name,
            'genome_range': (window_start, window_end),
            'seq': str(seq_obj),
            'score': measured_peak.score,
            'signalValue': measured_peak.signalValue,
            'is_binding': 1
        })

    wt_sequences_pos = pd.DataFrame.from_records(records)
    #wt_sequences_df = pd.concat([wt_sequences_df, wt_sequences_pos])
    all_pos_sequences.append(wt_sequences_pos)

    #Generate same amount of negative sequences that do not correlate to TF binding
    records_neg = []
    pos_seqs_ranges = pd.IntervalIndex.from_tuples(wt_sequences_pos['genome_range'])

    for i in range(len(wt_sequences_pos)):
      #To ensure it does not correlate to TF binding site regions
      found = False
      while not found:
        window_start = np.random.randint(0, len(genome_chr10['chr10']) - window_size)
        window_end = window_start + window_size
        current_window = pd.Interval(window_start, window_end)

        if not pos_seqs_ranges.overlaps(current_window).any():
          #add genomic sequences outside of tf binding regions
          seq_obj = genome_chr10['chr10'][window_start:window_end]
          records_neg.append({
            'tf_name': tf_name,
            'marker': seq_obj.name,
            'genome_range': (window_start, window_end),
            'seq': str(seq_obj),
            'score': 'NaN',
            'signalValue': None,
            'is_binding': 0 })

          found = True

    wt_sequences_neg = pd.DataFrame.from_records(records_neg)
    all_neg_sequences.append(wt_sequences_neg)


  wt_sequences_df = pd.concat(all_pos_sequences + all_neg_sequences, ignore_index=True)
  return wt_sequences_df



In [10]:
filepaths = {
    'ELF1': '/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/ELF1_peaks.bed',
    'STAT5A': '/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/STAT5A_peaks.bed',
    'STAT5B': '/content/drive/MyDrive/Colab Notebooks/cs690u_data/chip_seqs/STAT5B_peaks.bed'
    }

sequence_datasets = generate_allele_sequence_dataset_for_wt_gene(filepaths)


Analyzing peaks of ELF1...

ELF1 in Chr10 ChIP-seq Summary
Total peaks: 1,576
Mean peak width: 394.7 bp

Analyzing peaks of STAT5A...

STAT5A in Chr10 ChIP-seq Summary
Total peaks: 342
Mean peak width: 589.7 bp

Analyzing peaks of STAT5B...

STAT5B in Chr10 ChIP-seq Summary
Total peaks: 122
Mean peak width: 279.4 bp


/tmp/ipykernel_6758/18464913.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wt_sequences_df = pd.concat(all_pos_sequences + all_neg_sequences, ignore_index=True)


In [11]:
#VERIFICATION
#Generated dataset is balanced
sequence_datasets['is_binding'].value_counts()


,count
is_binding,
1,1627
0,1627


In [12]:
#VERIFICATION
#all sequences are the same length
set([len(s) for s in sequence_datasets['seq']])

{512}

Tokenize Sequences according to DNABERT-2 input format

In [13]:
def load_dnabert2_model():
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"\n✓ Device: {device}")

  # Load model
  print("\nLoading DNABERT-2...")
  tokenizer = AutoTokenizer.from_pretrained(
      "quietflamingo/dnabert2-fixed",
      trust_remote_code=True
  )

  config = BertConfig.from_pretrained("quietflamingo/dnabert2-fixed")
  model = AutoModel.from_pretrained("quietflamingo/dnabert2-fixed", config=config)
  model = model.to(device)

  if torch.cuda.is_available():
      model = model.half()

  print(f"✓ Model loaded on: {device}")
  return model, tokenizer, device

In [11]:
dnabert_model, tokenizer, device = load_dnabert2_model()


✓ Device: cuda

Loading DNABERT-2...


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

BertModel LOAD REPORT from: quietflamingo/dnabert2-fixed
Key                                                    | Status     | 
-------------------------------------------------------+------------+-
bert.encoder.layer.{0...11}.mlp.gated_layers.weight    | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.Wqkv.weight | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.Wqkv.bias   | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.wo.weight              | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.layernorm.bias         | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.wo.bias                | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.layernorm.weight       | UNEXPECTED | 
cls.predictions.decoder.weight                         | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight                 | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias               | UNEXPECTED | 
cls.predictions.deco

✓ Model loaded on: cuda


In [12]:
#Tokenize sequences according to DNABERT-2 input format (k-mer tokenization)
def batch_tokenize(sequences, batch_size=32):
    all_input_ids = []
    all_attention_masks = []

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]

        tokenized = tokenizer(
            batch,
            padding='max_length',
            truncation=True,
            max_length= 512,
            return_tensors="pt"
        )

        all_input_ids.append(tokenized['input_ids'])
        all_attention_masks.append(tokenized['attention_mask'])

    return {
        'input_ids': torch.cat(all_input_ids, dim=0),
        'attention_mask': torch.cat(all_attention_masks, dim=0),
    }

sequences = sequence_datasets['seq'].tolist()
tokenized = batch_tokenize(sequences, batch_size=32)

#Save
torch.save(tokenized, 'tokenized.pt')

# Load later
#tokenized = torch.load('tokenized.pt')


In [14]:
#easier to process data for training and evaluation if using pytorch dataset and dataloader


class DNASequenceDataset(Dataset):
    """
    PyTorch Dataset for DNA sequences and binding labels.
    """
    def __init__(self, sequences, labels, tokenizer, max_length=512):
        """
        Args:
            sequences: List of DNA sequences (strings)
            labels: List of labels (0 or 1)
            tokenizer: DNABERT-2 tokenizer
            max_length: Maximum sequence length
        """
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        """
        Returns a single training example.
        """
        sequence = self.sequences[idx]
        label = self.labels[idx]

        # Tokenize the sequence
        encoding = self.tokenizer(
            sequence,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),      # (seq_len,)
            'attention_mask': encoding['attention_mask'].squeeze(0),  # (seq_len,)
            'label': torch.tensor(label, dtype=torch.long)      # scalar
        }


Model Development

In [14]:
# class IL2RABindingClassifier(nn.Module):
#     """
#     Binary classifier for TF binding at IL2RA locus.

#     Input: DNA sequence (text)
#     Output: Probability of TF binding (0 or 1)
#     """

#     def __init__(self, dnabert_model, num_classes=2, dropout_rate=0.1, freeze_model=True):
#         super(IL2RABindingClassifier, self).__init__() #registers method with Pytorch's internal tracking
#         self.dnabert = dnabert_model
#         self.dropout = nn.Dropout(dropout_rate)
#         self.classifier = nn.Linear(768, num_classes)  # 768 → 2 #change to SVM

#         if freeze_model:
#             for param in self.dnabert.parameters():
#                 param.requires_grad = False

#     def forward(self, input_ids, attention_mask):
#         # Get frozen embeddings
#         with torch.no_grad():
#             outputs = self.dnabert(input_ids=input_ids, attention_mask=attention_mask)
#             cls_embedding = outputs.last_hidden_state[:, 0, :]  # (batch, 768)

#         # Trainable layers
#         x = self.dropout(cls_embedding)
#         logits = self.classifier(x)  # (batch, 2)

#         return logits

In [15]:
# #create classifier
# model = IL2RABindingClassifier(
#     dnabert_model= dnabert_model,
#     num_classes=2,
#     dropout_rate=0.1,
#     freeze_model=True  # Start with frozen DNABERT-2 (feature extraction)
# )

# #move to GPU
# model.to(device).float()

# print(f"IL2RA Binding Classifier Model on: {device}")

IL2RA Binding Classifier Model on: cuda


In [16]:
#VERIFICATION
#model.float()

# sample_dna_seq = "AAATCTTGTTGTAATCTATATTCAATTACCTTCTCTTCCAGTAACTGTCCTTTTTTAAAAATGCAATTACTAATAAAAAATTAAAAATGTGCTGAGTTTCACTTTTTCTTTTAGCTTCTTTGCCTATATTCTGTATGAATTTTCAAAATAGGTATAAAAAAACAAATGAATACAAATAAGAGAGGATAATCAGCCCCTGAATAACACTGTGTGTGGAAAATTAACACCAGGCTTCACTGGAGAAGCGGGACTGCGGACTGACTGATGGAAGGAGGAAGGTGACGATGCGGAAAGAAACGCGTCGCTTTCTATGTTTTGCTTTCTACGCATGGTAGTTGAGGATGTAGAACTTCCTGTGCTTGAAATTCTCAGTGCAGTTACATTAAAAGAAAAAAGGAAGTTGTGCCCTCTGAAGCTTTTCTTTCTTCTCCTTttttttttttggtttgttttttgagactgggtcttgctctgttgcccaggctggagtgcattggcacaatcatggctcactgcagcctcaaactcctatgcttgagcaatcctcctgcctcagcctccacagtagctgggaccacagatgcgaaccaccatgactgg"

# inputs = tokenizer(sample_dna_seq, return_tensors='pt', padding=True, truncation=True)
# inputs = {k: v.to(device) for k, v in inputs.items()}

# logits = model(inputs['input_ids'], inputs['attention_mask'])
# print(logits)


tensor([[ 0.1076, -0.6590]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [19]:
# #set up training components

# # Loss function
# criterion = nn.CrossEntropyLoss()

# # Optimizer (only train the classifier head initially)
# optimizer = Adam(model.classifier.parameters(), lr=1e-3)

# # Learning rate scheduler (optional but recommended)
# scheduler = ReduceLROnPlateau(
#     optimizer,
#     mode='max',        # Maximize metric (e.g., F1-score)
#     factor=0.5,        # Reduce LR by half
#     patience=3        # After 3 epochs without improvement
# )


In [20]:

# #train
# def train_epoch(model, dataloader, criterion, optimizer, device):
#     """
#     Train for one epoch.
#     """
#     model.train()  # Set to training mode

#     total_loss = 0
#     all_predictions = []
#     all_labels = []

#     for batch_idx, batch in enumerate(dataloader):
#         # Move batch to device
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['label'].to(device)

#         # Zero gradients
#         optimizer.zero_grad()

#         # Forward pass
#         logits = model(input_ids, attention_mask)

#         # Compute loss
#         loss = criterion(logits, labels)

#         # Backward pass
#         loss.backward()  # Compute gradients

#         # Update weights
#         optimizer.step()  # Apply gradients

#         # Track metrics
#         total_loss += loss.item()
#         predictions = torch.argmax(logits, dim=1).cpu().numpy()
#         all_predictions.extend(predictions)
#         all_labels.extend(labels.cpu().numpy())

#         # Print progress every 10 batches
#         if (batch_idx + 1) % 10 == 0:
#             print(f"  Batch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}")

#     # Compute epoch metrics
#     avg_loss = total_loss / len(dataloader)
#     accuracy = accuracy_score(all_labels, all_predictions)
#     f1 = f1_score(all_labels, all_predictions)

#     return avg_loss, accuracy, f1

# #validate
# def evaluate(model, dataloader, criterion, device):
#     """
#     Evaluate on validation/test set.
#     """
#     model.eval()  # Set to evaluation mode

#     total_loss = 0
#     all_predictions = []
#     all_labels = []
#     all_probs = []

#     with torch.no_grad():  # No gradient computation
#         for batch in dataloader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             labels = batch['label'].to(device)

#             # Forward pass
#             logits = model(input_ids, attention_mask)

#             # Compute loss
#             loss = criterion(logits, labels)
#             total_loss += loss.item()

#             # Get predictions and probabilities
#             probs = torch.softmax(logits, dim=1)
#             predictions = torch.argmax(logits, dim=1)

#             all_predictions.extend(predictions.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
#             all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of class 1

#     # Compute metrics
#     avg_loss = total_loss / len(dataloader)
#     accuracy = accuracy_score(all_labels, all_predictions)
#     f1 = f1_score(all_labels, all_predictions)

#     return avg_loss, accuracy, f1, all_predictions, all_probs

In [21]:
# sequences = sequence_datasets['seq'].tolist()
# labels = sequence_datasets['is_binding'].tolist()

# train_sequences, val_sequences, train_labels, val_labels = train_test_split(
#     sequences, labels,
#     test_size=0.2,
#     random_state=42,
#     stratify=labels
# )

# train_dataset = DNASequenceDataset(train_sequences, train_labels, tokenizer)
# val_dataset = DNASequenceDataset(val_sequences, val_labels, tokenizer)

# # Create dataloaders
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# #model init done before ...

# num_epochs = 10
# best_val_f1 = 0
# history = {
#     'train_loss': [],
#     'train_acc': [],
#     'train_f1': [],
#     'val_loss': [],
#     'val_acc': [],
#     'val_f1': []
# }

# for epoch in range(num_epochs):
#     print(f"\nEpoch {epoch + 1}/{num_epochs}")
#     print("-" * 40)

#     # Train
#     train_loss, train_acc, train_f1 = train_epoch(
#         model, train_loader, criterion, optimizer, device
#     )

#     # Validate
#     val_loss, val_acc, val_f1, _, _ = evaluate(
#         model, val_loader, criterion, device
#     )

#     # Save history
#     history['train_loss'].append(train_loss)
#     history['train_acc'].append(train_acc)
#     history['train_f1'].append(train_f1)
#     history['val_loss'].append(val_loss)
#     history['val_acc'].append(val_acc)
#     history['val_f1'].append(val_f1)

#     print(f"\nTrain Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
#     print(f"Val   Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

#     # Save best model
#     if val_f1 > best_val_f1:
#         best_val_f1 = val_f1
#         torch.save(model.state_dict(), 'best_model.pth')
#         print(f"✓ New best model saved! (Val F1: {val_f1:.4f})")


# print("\n" + "=" * 60)
# print(f"Training complete! Best Val F1: {best_val_f1:.4f}")
print("=" * 60)


Epoch 1/10
----------------------------------------
  Batch 10/326, Loss: 0.7398
  Batch 20/326, Loss: 0.6155
  Batch 30/326, Loss: 0.5354
  Batch 40/326, Loss: 0.7982
  Batch 50/326, Loss: 0.7943
  Batch 60/326, Loss: 0.8343
  Batch 70/326, Loss: 0.5809
  Batch 80/326, Loss: 0.7645
  Batch 90/326, Loss: 0.3455
  Batch 100/326, Loss: 0.5077
  Batch 110/326, Loss: 0.5766
  Batch 120/326, Loss: 0.7301
  Batch 130/326, Loss: 0.6986
  Batch 140/326, Loss: 0.5661
  Batch 150/326, Loss: 0.6504
  Batch 160/326, Loss: 0.9694
  Batch 170/326, Loss: 0.5685
  Batch 180/326, Loss: 0.6230
  Batch 190/326, Loss: 0.4691
  Batch 200/326, Loss: 0.5579
  Batch 210/326, Loss: 0.4469
  Batch 220/326, Loss: 0.6277
  Batch 230/326, Loss: 0.6799
  Batch 240/326, Loss: 0.8518
  Batch 250/326, Loss: 0.5647
  Batch 260/326, Loss: 0.8869
  Batch 270/326, Loss: 0.5535
  Batch 280/326, Loss: 0.7221
  Batch 290/326, Loss: 0.5229
  Batch 300/326, Loss: 1.0050
  Batch 310/326, Loss: 0.5480
  Batch 320/326, Loss: 0.8

In [33]:
class IL2RABindingClassifier(nn.Module):
    """
    Binary classifier for TF binding prediction using DNABERT-2 + Logistic Regression.

    Architecture:
    - DNABERT-2 (frozen): Extracts 768-dim embeddings
    - Logistic Regression (sklearn): Classifies embeddings into binding/no-binding
    """

    def __init__(self, dnabert_model, lr_classifier=None, dropout_rate=0.1):
        """
        Args:
            dnabert_model: Pre-trained DNABERT-2 model
            lr_classifier: Pre-trained sklearn LogisticRegression (optional)
            dropout_rate: Dropout rate for regularization
        """
        super(IL2RABindingClassifier, self).__init__()

        self.dnabert = dnabert_model
        self.dropout = nn.Dropout(dropout_rate)

        # Logistic Regression classifier (sklearn)
        self.lr = lr_classifier
        self.lr_trained = lr_classifier is not None

        # Freeze DNABERT-2
        for param in self.dnabert.parameters():
            param.requires_grad = False

    def extract_embeddings(self, input_ids, attention_mask):
        """
        Extract DNABERT-2 embeddings.

        Args:
            input_ids: Token IDs (batch_size, seq_len)
            attention_mask: Attention mask (batch_size, seq_len)

        Returns:
            embeddings: (batch_size, 768) numpy array
        """
        # Get DNABERT-2 output
        with torch.no_grad():
            outputs = self.dnabert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            cls_embedding = outputs.last_hidden_state[:, 0, :]  # (batch_size, 768)

        # Apply dropout
        cls_embedding = self.dropout(cls_embedding)

        # Convert to numpy for sklearn
        embeddings = cls_embedding.cpu().detach().numpy()

        return embeddings

    def forward(self, input_ids, attention_mask):
        """
        Forward pass through DNABERT-2 + Logistic Regression.

        Args:
            input_ids: Token IDs (batch_size, seq_len)
            attention_mask: Attention mask (batch_size, seq_len)

        Returns:
            predictions: (batch_size,) tensor of class predictions
            probabilities: (batch_size, 2) tensor of class probabilities
        """
        # Extract embeddings
        embeddings = self.extract_embeddings(input_ids, attention_mask)

        # Check if LR is trained
        if self.lr is None or not self.lr_trained:
            raise RuntimeError(
                "Logistic Regression classifier not trained yet! "
                "Call train_classifier() first."
            )

        # Predict with Logistic Regression
        predictions = self.lr.predict(embeddings)
        probabilities = self.lr.predict_proba(embeddings)

        # Convert back to PyTorch tensors
        predictions = torch.tensor(predictions, dtype=torch.long)
        probabilities = torch.tensor(probabilities, dtype=torch.float32)

        return predictions, probabilities

    def train_classifier(self, train_loader, device, lr_params=None):
        """
        Train the Logistic Regression classifier on extracted embeddings.

        Args:
            train_loader: PyTorch DataLoader with training data
            device: torch.device (cuda/cpu)
            lr_params: dict of Logistic Regression hyperparameters

        Returns:
            Trained LogisticRegression classifier
        """
        print("=" * 60)
        print("Training Logistic Regression Classifier")
        print("=" * 60)

        print("\n1. Extracting embeddings from training data...")

        # Extract all embeddings and labels
        all_embeddings = []
        all_labels = []

        self.eval()  # Set to eval mode

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']

            # Extract embeddings
            embeddings = self.extract_embeddings(input_ids, attention_mask)

            all_embeddings.append(embeddings)
            all_labels.append(labels.numpy())

            if (batch_idx + 1) % 10 == 0:
                print(f"   Processed {batch_idx + 1}/{len(train_loader)} batches")

        # Concatenate all batches
        X_train = np.vstack(all_embeddings)  # (N, 768)
        y_train = np.concatenate(all_labels)  # (N,)

        print(f"\n2. Embeddings extracted:")
        print(f"   ✓ Shape: {X_train.shape}")
        print(f"   ✓ Samples: {X_train.shape[0]}")
        print(f"   ✓ Class distribution: {np.bincount(y_train)}")

        # Set default Logistic Regression parameters
        if lr_params is None:
            lr_params = {
                'C': 1.0,                    # Regularization strength
                'penalty': 'l2',              # L2 regularization
                'solver': 'lbfgs',            # Optimization algorithm
                'max_iter': 1000,             # Maximum iterations
                'class_weight': 'balanced',   # Handle class imbalance
                'random_state': 42,
                'verbose': 0
            }

        print(f"\n3. Training Logistic Regression...")
        print(f"   Parameters: {lr_params}")

        # Create and train Logistic Regression
        self.lr = LogisticRegression(**lr_params)
        self.lr.fit(X_train, y_train)
        self.lr_trained = True

        # Evaluate on training set
        train_predictions = self.lr.predict(X_train)
        train_accuracy = np.mean(train_predictions == y_train)

        train_f1 = f1_score(y_train, train_predictions)

        print(f"\n4. Training Results:")
        print(f"   ✓ Training Accuracy: {train_accuracy:.4f}")
        print(f"   ✓ Training F1-score: {train_f1:.4f}")
        print(f"   ✓ Number of iterations: {self.lr.n_iter_[0]}")

        # Show coefficients info
        print(f"\n5. Model Info:")
        print(f"   ✓ Number of features: {self.lr.coef_.shape[1]}")
        print(f"   ✓ Intercept: {self.lr.intercept_}")

        print("\n" + "=" * 60)
        print("✓ Logistic Regression training complete!")
        print("=" * 60)

        return self.lr

    def evaluate(self, val_loader, device):
        """
        Evaluate the classifier on validation/test data.

        Args:
            val_loader: PyTorch DataLoader with validation data
            device: torch.device (cuda/cpu)

        Returns:
            metrics: Dictionary of evaluation metrics
        """
        from sklearn.metrics import (
            accuracy_score, f1_score, precision_score,
            recall_score, confusion_matrix, classification_report
        )

        print("\nEvaluating on validation set...")

        # Extract embeddings and labels
        all_embeddings = []
        all_labels = []

        self.eval()

        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']

            embeddings = self.extract_embeddings(input_ids, attention_mask)

            all_embeddings.append(embeddings)
            all_labels.append(labels.numpy())

        X_val = np.vstack(all_embeddings)
        y_val = np.concatenate(all_labels)

        # Predict
        predictions = self.lr.predict(X_val)
        probabilities = self.lr.predict_proba(X_val)

        # Calculate metrics
        metrics = {
            'accuracy': accuracy_score(y_val, predictions),
            'f1': f1_score(y_val, predictions),
            'precision': precision_score(y_val, predictions),
            'recall': recall_score(y_val, predictions),
            'roc_auc': roc_auc_score(y_val, probabilities[:, 1]),
            'confusion_matrix': confusion_matrix(y_val, predictions),
            'predictions': predictions,
            'probabilities': probabilities,
            'true_labels': y_val
        }

        print(f"\n✓ Validation Accuracy: {metrics['accuracy']:.4f}")
        print(f"✓ Validation F1-score: {metrics['f1']:.4f}")
        print(f"✓ Precision: {metrics['precision']:.4f}")
        print(f"✓ Recall: {metrics['recall']:.4f}")
        print(f"✓ ROC AUC: {metrics['roc_auc']:.4f}")

        print("\nClassification Report:")
        print(classification_report(
            y_val, predictions,
            target_names=['No Binding', 'Binding']
        ))

        print("\nConfusion Matrix:")
        print(metrics['confusion_matrix'])

        return metrics



Training Model

In [34]:
dnabert_model, tokenizer, device = load_dnabert2_model()
model = IL2RABindingClassifier(dnabert_model, dropout_rate=0.1)

model = model.to(device)
#model.float() #need this?


✓ Device: cuda

Loading DNABERT-2...


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

BertModel LOAD REPORT from: quietflamingo/dnabert2-fixed
Key                                                    | Status     | 
-------------------------------------------------------+------------+-
bert.encoder.layer.{0...11}.mlp.gated_layers.weight    | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.Wqkv.weight | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.Wqkv.bias   | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.wo.weight              | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.layernorm.bias         | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.wo.bias                | UNEXPECTED | 
bert.encoder.layer.{0...11}.mlp.layernorm.weight       | UNEXPECTED | 
cls.predictions.decoder.weight                         | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight                 | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias               | UNEXPECTED | 
cls.predictions.deco

✓ Model loaded on: cuda


In [35]:
sequences = sequence_datasets['seq'].tolist()
labels = sequence_datasets['is_binding'].tolist()

#split into train and test sets
train_sequences, val_sequences, train_labels, val_labels = train_test_split(
    sequences, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_dataset = DNASequenceDataset(train_sequences, train_labels, tokenizer)
val_dataset = DNASequenceDataset(val_sequences, val_labels, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [36]:
print("\n4. Training classifier...")

lr_params = {
    'C': 1.0,
    'penalty': 'l2',
    'solver': 'lbfgs',
    'max_iter': 1000,
    'class_weight': 'balanced',
    'random_state': 42
}

model.train_classifier(train_loader, device, lr_params=lr_params)


4. Training classifier...
Training Logistic Regression Classifier

1. Extracting embeddings from training data...
   Processed 10/326 batches
   Processed 20/326 batches
   Processed 30/326 batches
   Processed 40/326 batches
   Processed 50/326 batches
   Processed 60/326 batches
   Processed 70/326 batches
   Processed 80/326 batches
   Processed 90/326 batches
   Processed 100/326 batches
   Processed 110/326 batches
   Processed 120/326 batches
   Processed 130/326 batches
   Processed 140/326 batches
   Processed 150/326 batches
   Processed 160/326 batches
   Processed 170/326 batches
   Processed 180/326 batches
   Processed 190/326 batches
   Processed 200/326 batches
   Processed 210/326 batches
   Processed 220/326 batches
   Processed 230/326 batches
   Processed 240/326 batches
   Processed 250/326 batches
   Processed 260/326 batches
   Processed 270/326 batches
   Processed 280/326 batches
   Processed 290/326 batches
   Processed 300/326 batches
   Processed 310/326 bat

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [37]:
print("\n5. Evaluating on validation set...")

metrics = model.evaluate(val_loader, device)


5. Evaluating on validation set...

Evaluating on validation set...

✓ Validation Accuracy: 0.7389
✓ Validation F1-score: 0.7157
✓ Precision: 0.7839
✓ Recall: 0.6585
✓ ROC AUC: 0.7951

Classification Report:
              precision    recall  f1-score   support

  No Binding       0.71      0.82      0.76       326
     Binding       0.78      0.66      0.72       325

    accuracy                           0.74       651
   macro avg       0.75      0.74      0.74       651
weighted avg       0.75      0.74      0.74       651


Confusion Matrix:
[[267  59]
 [111 214]]


In [31]:
torch.save(model.state_dict(), 'best_model_logreg.pth')

Compare with sklearn technique of feature extraction -> training classifier on data

Stage 1: Extract embeddings with DNABERT-2 (frozen)
   DNA sequences → DNABERT-2 → 768-dim embeddings

Stage 2: Train sklearn classifier on embeddings
   Embeddings + labels → SVM/LogisticRegression → Predictions

Extract Embeddings

In [ ]:
with torch.no_grad():
    outputs = model(**tokenized)  # tokenized from before
    embeddings = outputs.last_hidden_state.mean(dim=1)

# Now you have embeddings!
print(embeddings.shape)  # [num_sequences, 768]

For prediction

In [20]:
#finding gene variants that correlate to SLE

#lit review: https://pmc.ncbi.nlm.nih.gov/articles/PMC2662820/
#variant rs11594656 #https://genopedia.com/variant/11594656 #https://www.ncbi.nlm.nih.gov/clinvar/RCV000015782/

#lit review: https://journals.plos.org/plosgenetics/article?id=10.1371/journal.pgen.1002406#pgen.1002406-Gregersen1
# variant Rs11101442 (pos 49.606 Mb) specifically on chr10  #https://www.ebi.ac.uk/gwas/variants/rs11101442

lupus_variants = {
    'rs11594656': {'ref': 'T', 'alternate': ['A'], 'pos': 6080046},
    'rs11101442': {'ref': 'C', 'alternate': ['A', 'T'], 'pos': 48728291}
    }

In [17]:

#took a look at farh et al SNP table
#inconclusive...SNPs here are associated with genes that are genomically independent of IL2RA


farh_df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/cs690u_data/41586_2015_BFnature13835_MOESM8_ESM.xls")

farh_sle_df = farh_df[(farh_df['Disease'] == 'Systemic_lupus_erythematosus') & (farh_df['chr'] == 'chr10')]
farh_sle_df

,Disease,IndexSNP_riskAllele,SNP,chr,pos,PICS_probability,Annotation,nearestGene,eQTL,eQTLdirection,...,colonic_mucosa,Duodenum_mucosa,AdiposeNuclei,HepG2,Liver1,Pancreatic_Islets,kidney,HSMM-myotube,NH-Osteoblast,chondrogenic_dif_cells
3024,Systemic_lupus_erythematosus,rs1913517-A,rs1913517,chr10,50119054,0.2286,none,WDFY4,none,NaN,...,0,0,0,0,0,0,0,0,0,0
3025,Systemic_lupus_erythematosus,rs1913517-A,rs7068839,chr10,50110015,0.2286,none,WDFY4,none,NaN,...,1,1,1,0,1,0,0,0,0,0
3026,Systemic_lupus_erythematosus,rs1913517-A,rs7069142,chr10,50110232,0.2286,none,WDFY4,none,NaN,...,0,0,0,0,0,0,0,0,0,0
3027,Systemic_lupus_erythematosus,rs1913517-A,rs6537583,chr10,50120707,0.2286,none,WDFY4,none,NaN,...,0,0,0,0,0,0,0,0,0,0
3028,Systemic_lupus_erythematosus,rs4948496-C,rs4948496,chr10,63805617,0.2094,none,ARID5B,none,NaN,...,1,1,1,1,1,1,1,1,1,1
3029,Systemic_lupus_erythematosus,rs4948496-C,rs6479781,chr10,63805735,0.2094,none,ARID5B,none,NaN,...,1,1,1,1,1,1,1,1,1,1
3030,Systemic_lupus_erythematosus,rs4948496-C,rs12355313,chr10,63805376,0.1031,none,ARID5B,none,NaN,...,1,1,1,1,1,1,1,1,1,1
3031,Systemic_lupus_erythematosus,rs4948496-C,rs12357548,chr10,63803472,0.1028,none,ARID5B,none,NaN,...,1,1,1,1,1,1,1,1,1,1
3032,Systemic_lupus_erythematosus,rs4948496-C,rs10821949,chr10,63811678,0.0638,none,ARID5B,none,NaN,...,1,1,1,1,1,1,1,1,1,1
3033,Systemic_lupus_erythematosus,rs4948496-C,rs10821950,chr10,63811967,0.0636,none,ARID5B,none,NaN,...,1,1,1,1,1,1,1,1,1,1


Look at Variant to Analyze and Predict Using Model

In [18]:
#adding common variants in IL2RA region
#https://gnomad.broadinstitute.org/gene/ENSG00000134460?dataset=gnomad_r4

gnomAD_vcf_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/cs690u_data/gnomAD_v4.1.1_ENSG00000134460_2026_05_04_01_58_32.csv")
gnomAD_vcf_df = gnomAD_vcf_df[gnomAD_vcf_df['Flags'] != 'lof_flag']
gnomAD_risky_df = gnomAD_vcf_df[gnomAD_vcf_df['ClinVar Germline Classification'].isin(['Pathogenic', 'Likely pathogenic'])]
gnomAD_healthy_df = gnomAD_vcf_df[gnomAD_vcf_df['ClinVar Germline Classification'].isin(['Likely benign', 'Benign/Likely benign', 'Benign'])]

In [22]:
for index, row in gnomAD_risky_df.iterrows():
  metadata = {
      'ref': row['Reference'],
      'alternate': row['Alternate'],
      'pos': row['Position']
  }

  lupus_variants[row['rsIDs']] = metadata

#pull sequence for wt and mutant
for variant_id, metadata in lupus_variants.items():
  window_start = metadata['pos'] - 256
  window_end = metadata['pos'] + 256
  genome_chr10seq_obj = genome_chr10['chr10'][window_start:window_end]
  wt_seq = str(genome_chr10seq_obj.seq)
  mut_seq = []

{'rs11594656': {'ref': 'T', 'alternate': ['A'], 'pos': 6080046}, 'rs11101442': {'ref': 'C', 'alternate': ['A', 'T'], 'pos': 48728291}, 'rs1250584991': {'ref': 'C', 'alternate': 'T', 'pos': 6019869}}
